In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.preprocessing import LabelEncoder
from recsys_pipeliner.recommendations.transformer import (
    SimilarityTransformer,
    UserItemMatrixTransformer,
)
from recsys_pipeliner.algorithms.recommenders import ItemBasedCFRecommender
from IPython.display import display
from recsys_pipeliner.dataset import RatingsDataset

In [2]:
# load test data
data_types = {"user_id": str, "item_id": str, "rating": np.float64}
user_item_ratings = pd.read_csv(
    "../../tests/test_data/user_item_ratings_toy.csv", dtype=data_types
)

display(user_item_ratings.head(3))

# encode the user/item ids
item_encoder = LabelEncoder()
user_encoder = LabelEncoder()

user_item_ratings["item_id"] = item_encoder.fit_transform(user_item_ratings["item_id"])
user_item_ratings["user_id"] = user_encoder.fit_transform(user_item_ratings["user_id"])

unique_users = pd.Series(user_encoder.classes_)
unique_items = pd.Series(item_encoder.classes_)

print("unique_users.shape", unique_users.shape)
print("unique_items.shape", unique_items.shape)

display(user_item_ratings.head(3))

# create the user/item matrix
user_item_matrix_transformer = UserItemMatrixTransformer()

user_item_matrix = user_item_matrix_transformer.transform(
    user_item_ratings.to_numpy(),
)

print("user_item_matrix.shape", user_item_matrix.shape)

# sanity check
users = user_item_ratings["user_id"].to_numpy().astype(int)
items = user_item_ratings["item_id"].to_numpy().astype(int)
ratings = user_item_ratings["rating"].to_numpy().astype(np.float32)
for user, item, rating in zip(users, items, ratings):
    assert user_item_matrix[user, item] == rating

,user_id,item_id,rating
0,U00001,I00024,0.8
1,U00001,I00013,0.6
2,U00001,I00005,1.0


unique_users.shape (12,)
unique_items.shape (24,)


,user_id,item_id,rating
0,0,23,0.8
1,0,12,0.6
2,0,4,1.0


user_item_matrix.shape (12, 24)


In [3]:
dataset = RatingsDataset(user_item_ratings)
loo_iterator = dataset.leave_one_out()

print("user_item_ratings", user_item_ratings.shape)
for trainset, testset in loo_iterator:
    print("trainset", trainset.shape)
    print("testset", testset.shape)


user_item_ratings (96, 3)
trainset (84, 3)
testset (12, 3)
